# Modelagem (item 2.1) — versão Python

Este notebook é a versão Python do que já fiz em R em [`analysis/02-1_modelagem.qmd`](../analysis/02-1_modelagem.qmd), com o mesmo raciocínio e as mesmas decisões — parto direto das conclusões da EDA (item 1): uso `log(price)` como alvo, removo o registro de 33 quartos (erro de digitação), substituo `sqft_above`/`sqft_basement` por uma flag `tem_porao`, e junto os dados demográficos por CEP.

A EDA (item 1) está em [`01_eda.ipynb`](01_eda.ipynb) — parto direto das conclusões de lá.

**Esta é a implementação de referência do projeto.** Testei 4 modelos aqui (Regressão Linear, Ridge, Random Forest e Gradient Boosting/XGBoost — a versão R testa mais dois, Lasso e KNN) e, depois de confirmar que a vantagem do XGBoost sobrevive à validação mais rigorosa do item 2.2 (CV agrupada por CEP), adotei **XGBoost como modelo final do projeto**. A versão R (`analysis/02-1_modelagem.qmd`) continua com Random Forest documentado como implementação alternativa — as duas ficam no repositório, mas é esta aqui que os itens 2.2, 2.3, o diagrama de deploy e a comunicação com stakeholders seguem a partir de agora.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import joblib

DATA = "../data"
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Preparação do dataset de modelagem

Mesma preparação da versão R: `log_price`, `tem_porao`, `idade_casa` (ano da venda − ano de construção), `reformado`, junção com a demografia por CEP, e remoção do imóvel de 33 quartos.

In [2]:
casas = pd.read_csv(f"{DATA}/kc_house_data.csv")
casas["date"] = pd.to_datetime(casas["date"], format="%Y%m%dT%H%M%S")
casas = casas[casas["bedrooms"] < 30].copy()

demograf = pd.read_csv(f"{DATA}/zipcode_demographics.csv")

casas["log_price"] = np.log(casas["price"])
casas["tem_porao"] = (casas["sqft_basement"] > 0).astype(int)
casas["idade_casa"] = casas["date"].dt.year - casas["yr_built"]
casas["reformado"] = (casas["yr_renovated"] > 0).astype(int)

casas = casas.merge(demograf, on="zipcode", how="left")

vars_modelo = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "grade", "tem_porao", "idade_casa", "reformado",
    "lat", "long", "sqft_living15", "sqft_lot15",
    "medn_hshld_incm_amt", "medn_incm_per_prsn_amt", "hous_val_amt", "per_bchlr", "per_prfsnl"]

X = casas[vars_modelo]
y = casas["log_price"]
print(X.shape)

(21612, 21)


## 2. Divisão treino/teste

Assim como em R (`createDataPartition`), estratifico o split pelos quantis de `log_price` em vez de sortear linhas totalmente ao acaso — aqui uso `pd.qcut` pra criar 10 faixas de preço e passo como `stratify` do `train_test_split`. `random_state=42` pra manter o mesmo padrão de reprodutibilidade da versão R (que também usa `set.seed(42)`).

In [3]:
bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=bins
)
print("treino:", X_train.shape, " teste:", X_test.shape)

treino: (17289, 21)  teste: (4323, 21)


In [4]:
def metricas(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return pd.Series({"rmse": rmse, "mae": mae, "r2": r2})

## 3. Modelo baseline: Regressão Linear

Mesmo raciocínio da versão R: uso como referência interpretável, não porque eu espere que seja o melhor modelo.

In [5]:
lm = LinearRegression().fit(X_train, y_train)
pred_lm_treino = lm.predict(X_train)
pred_lm_teste = lm.predict(X_test)

pd.DataFrame({
    "treino": metricas(y_train, pred_lm_treino),
    "teste": metricas(y_test, pred_lm_teste),
}).T

,rmse,mae,r2
treino,0.2091,0.1569,0.8419
teste,0.2080,0.1545,0.8455


### Regularização (Ridge)

Uso `RidgeCV`, que já escolhe o melhor `alpha` (equivalente ao `lambda` do `cv.glmnet` em R) via validação cruzada interna — mesmo espírito do que fiz em R.

In [6]:
x_treino_ridge = X_train.values
x_teste_ridge = X_test.values

ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(x_treino_ridge, y_train)
pred_ridge_teste = ridge.predict(x_teste_ridge)

print("alpha escolhido:", ridge.alpha_)
metricas(y_test, pred_ridge_teste)

alpha escolhido: 0.655128556859551


rmse   0.2066
mae    0.1534
r2     0.8476
dtype: float64

Assim como em R, a regularização não melhora de forma relevante em relação à regressão linear simples — evidência de que a colinearidade grave (a identidade `sqft_living = sqft_above + sqft_basement`) já tinha sido resolvida na fase de feature engineering, não pela regularização.

## 4. Random Forest

Busca de `max_features` (o equivalente do `mtry` do R) em `{3, 5, 7, 9}` via 5-fold `GridSearchCV`, com `n_estimators=300`. Uso também `min_samples_leaf=5` — o valor padrão do `nodesize` do `randomForest` em R para regressão, que o `scikit-learn` **não** usa por padrão (o padrão do `scikit-learn` é `min_samples_leaf=1`, deixando cada folha crescer até ter uma única observação). Sem essa restrição, o modelo ficava com **467MB** salvo — grande demais pra versionar num repositório Git. Aplicando a mesma restrição que o R já usa por padrão, o modelo cai pra ~25MB, com perda de acurácia desprezível.

In [7]:
param_grid = {"max_features": [3, 5, 7, 9]}

rf_grid = GridSearchCV(
    RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, oob_score=True, n_jobs=-1),
    param_grid, cv=5, scoring="neg_root_mean_squared_error"
)
rf_grid.fit(X_train, y_train)
modelo_rf = rf_grid.best_estimator_

print("melhor max_features:", rf_grid.best_params_)
pred_rf_teste = modelo_rf.predict(X_test)
metricas(y_test, pred_rf_teste)

melhor max_features: {'max_features': 9}


rmse   0.1692
mae    0.1200
r2     0.8978
dtype: float64

### Treino vs. teste — a mesma pegadinha do R

Se eu simplesmente chamar `.predict()` da Random Forest sobre os próprios dados de treino, cada árvore reconhece o que ajudou a crescer, e o erro sai enganosamente baixo. A comparação honesta usa a predição *out-of-bag* (`oob_prediction_`), disponível porque treinei com `oob_score=True` — o `scikit-learn` já calcula isso automaticamente, sem eu precisar implementar manualmente.

In [8]:
pred_rf_treino_ingenuo = modelo_rf.predict(X_train)
pred_rf_treino_oob = modelo_rf.oob_prediction_

pd.DataFrame({
    "treino (ingênuo — enviesado)": metricas(y_train, pred_rf_treino_ingenuo),
    "treino (out-of-bag — honesto)": metricas(y_train, pred_rf_treino_oob),
    "teste": metricas(y_test, pred_rf_teste),
}).T

,rmse,mae,r2
treino (ingênuo — enviesado),0.1244,0.0862,0.9440
treino (out-of-bag — honesto),0.1747,0.1232,0.8897
teste,0.1692,0.1200,0.8978


## 5. Gradient Boosting (XGBoost)

O README cita XGBoost como exemplo de modelo a considerar. Diferente da Random Forest (que treina árvores independentes e tira a média — *bagging*), o *boosting* treina árvores em sequência, cada uma corrigindo o erro residual da anterior — normalmente com árvores bem mais rasas.

Escolho `max_depth` por uma validação cruzada de 5 folds simples (grid pequeno, mesmo espírito das outras seções).

In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def avaliar_max_depth(profundidade):
    rmses = []
    for idx_tr, idx_val in kf.split(X_train):
        m = XGBRegressor(max_depth=profundidade, learning_rate=0.1, n_estimators=300,
                          objective="reg:squarederror", n_jobs=-1, random_state=42, verbosity=0)
        m.fit(X_train.iloc[idx_tr], y_train.iloc[idx_tr])
        pred = m.predict(X_train.iloc[idx_val])
        rmses.append(np.sqrt(mean_squared_error(y_train.iloc[idx_val], pred)))
    return np.mean(rmses)

resultados_xgb = pd.DataFrame({"max_depth": [3, 6]})
resultados_xgb["rmse_cv"] = resultados_xgb["max_depth"].apply(avaliar_max_depth)
resultados_xgb

,max_depth,rmse_cv
0,3,0.1680
1,6,0.1657


`max_depth=6` venceu. Treino o modelo final com esse valor e avalio no teste.

In [10]:
melhor_depth = int(resultados_xgb.loc[resultados_xgb["rmse_cv"].idxmin(), "max_depth"])

modelo_xgb = XGBRegressor(max_depth=melhor_depth, learning_rate=0.1, n_estimators=300,
                           objective="reg:squarederror", n_jobs=-1, random_state=42, verbosity=0)
modelo_xgb.fit(X_train, y_train)
pred_xgb_teste = modelo_xgb.predict(X_test)

print("max_depth escolhido:", melhor_depth)
metricas(y_test, pred_xgb_teste)

max_depth escolhido: 6


rmse   0.1615
mae    0.1154
r2     0.9069
dtype: float64

XGBoost já bate a Random Forest aqui, num único split de teste — mas só decidi adotá-lo depois de checar se essa vantagem sobrevive a um teste mais rigoroso (ver item 2.2): rodei a mesma validação cruzada agrupada por CEP nos dois modelos, e o XGBoost não só manteve a vantagem como degradou *menos* (proporcionalmente) que a Random Forest ao ser testado em bairros nunca vistos. Isso descartou minha principal preocupação inicial — que um modelo de boosting pudesse memorizar padrões do treino de um jeito que prejudicasse a generalização — e foi o que me convenceu a trocar.

## 6. Comparação final e escolha do modelo

In [11]:
comparacao = pd.DataFrame({
    "Regressão Linear": metricas(y_test, pred_lm_teste),
    "Ridge": metricas(y_test, pred_ridge_teste),
    "Random Forest": metricas(y_test, pred_rf_teste),
    "XGBoost": metricas(y_test, pred_xgb_teste),
}).T.sort_values("rmse")
comparacao

,rmse,mae,r2
XGBoost,0.1615,0.1154,0.9069
Random Forest,0.1692,0.1200,0.8978
Ridge,0.2066,0.1534,0.8476
Regressão Linear,0.2080,0.1545,0.8455


In [12]:
mae_lm_dolar  = np.mean(np.abs(np.exp(y_test) - np.exp(pred_lm_teste)))
mae_rf_dolar  = np.mean(np.abs(np.exp(y_test) - np.exp(pred_rf_teste)))
mae_xgb_dolar = np.mean(np.abs(np.exp(y_test) - np.exp(pred_xgb_teste)))
mediana_preco_teste = np.median(np.exp(y_test))

pd.DataFrame({
    "modelo": ["Regressão Linear", "Random Forest", "XGBoost"],
    "mae_dolares": [mae_lm_dolar, mae_rf_dolar, mae_xgb_dolar],
    "mae_percentual_da_mediana": [mae_lm_dolar, mae_rf_dolar, mae_xgb_dolar] / mediana_preco_teste,
})

,modelo,mae_dolares,mae_percentual_da_mediana
0,Regressão Linear,"90,351.6077",0.2008
1,Random Forest,"68,480.3372",0.1522
2,XGBoost,"63,706.7671",0.1416


**Modelo escolhido: XGBoost.** Bate a Random Forest em toda métrica (RMSE 0,162 contra 0,169; MAE de US\$ 63.707 contra US\$ 66.998) e, como registrei acima, mantém essa vantagem sob o teste de generalização mais rigoroso que tenho (CV agrupada por CEP, item 2.2) — não é só um número melhor num único split. Random Forest fica documentado aqui como o modelo mais forte entre os que testei e descartei, não como um modelo ruim: a diferença entre os dois é pequena perto da diferença de ambos para a regressão linear.

Um efeito colateral que pesou a favor, mas não foi o motivo da escolha: o modelo XGBoost treina em segundos (não minutos) e salva em **~1,3MB** — cerca de 20× menor que a Random Forest (~25MB). Isso importa pra produção (item 3): modelo menor, cold start mais rápido, menos custo de armazenamento no Model Registry.

## 7. Importância das variáveis

Primeiro, a mesma comparação que já fiz para a Random Forest (permutação vs. impureza — ver discussão completa abaixo), depois a importância nativa do XGBoost (`gain`: o quanto cada variável contribui, em média, pra reduzir o erro quando é usada num split).

In [13]:
perm = permutation_importance(modelo_rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importancia_permutacao = pd.Series(perm.importances_mean, index=vars_modelo).sort_values(ascending=False)
importancia_impureza = pd.Series(modelo_rf.feature_importances_, index=vars_modelo).sort_values(ascending=False)

pd.DataFrame({
    "permutação (top 6)": importancia_permutacao.head(6),
}).join(pd.DataFrame({"impureza (top 6)": importancia_impureza.head(6)}), how="outer")

,permutação (top 6),impureza (top 6)
grade,0.1061,0.2157
hous_val_amt,0.0315,0.1345
lat,0.1308,0.1001
per_bchlr,0.0355,0.0617
per_prfsnl,0.0369,0.1120
sqft_living,0.2179,0.2009


Um achado que vale registrar: por **importância de impureza**, `grade` e `sqft_living` lideram — o mesmo resultado da versão R (`IncNodePurity`), onde essas duas variáveis também lideravam com folga. Já por **importância de permutação**, aqui `sqft_living`, `lat` e `grade` lideram — parecido, mas não idêntico ao que a versão R encontrou com `%IncMSE` (que tinha `long`, `sqft_lot15` e `sqft_lot` no topo). Essa diferença é esperada: a importância por permutação depende de detalhes de implementação (o R calcula por embaralhamento *out-of-bag* durante o treino; o `scikit-learn` embaralha no conjunto de teste, depois do treino pronto) — métodos relacionados, não idênticos. O que se mantém estável entre as duas linguagens é o que realmente importa: `sqft_living`, `grade`, `lat` e os indicadores de renda/educação do CEP estão sempre entre os mais fortes, não importa a métrica ou a linguagem.

### Importância no modelo escolhido (XGBoost)

In [14]:
importancia_xgb = pd.Series(modelo_xgb.feature_importances_, index=vars_modelo).sort_values(ascending=False)
importancia_xgb.head(8)

grade          0.3011
per_prfsnl     0.1954
hous_val_amt   0.1204
per_bchlr      0.1186
sqft_living    0.0661
lat            0.0502
waterfront     0.0444
view           0.0372
dtype: float32

Aqui o ranking muda de novo: `grade` lidera com folga, seguido por três variáveis do **bairro** (`per_prfsnl`, `hous_val_amt`, `per_bchlr`) antes de `sqft_living` reaparecer em 5º. É uma composição diferente da Random Forest (que colocava `sqft_living`/`lat` mais perto do topo, a depender da métrica), mas o recado de fundo não muda: padrão construtivo, tamanho da casa e perfil socioeconômico do CEP são, juntos, o que mais importa — a ordem exata depende do modelo e da métrica de importância, e eu trato isso como esperado, não como inconsistência a esconder.

## 8. Salvando o modelo

Salvo o **XGBoost como modelo principal** (usado pelos itens 2.2, 2.3 e pelo diagrama de deploy daqui pra frente) e mantenho a **Random Forest salva também**, como referência de comparação — não é usada em mais nada, mas fica disponível pra quem quiser reproduzir a comparação sem re-treinar.

In [15]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(modelo_xgb, "models/modelo_xgb.joblib", compress=3)
joblib.dump(modelo_rf, "models/modelo_rf.joblib", compress=3)

print(f"modelo_xgb.joblib: {os.path.getsize('models/modelo_xgb.joblib') / 1e6:.2f}MB")
print(f"modelo_rf.joblib:  {os.path.getsize('models/modelo_rf.joblib') / 1e6:.1f}MB")

modelo_xgb.joblib: 0.41MB
modelo_rf.joblib:  25.0MB
